# Project 04: Chatbot - Complete Solution 🤖

---

## Phase 1: Setup

In [ ]:
import numpy as np
import pandas as pd
import json
import random
from datetime import datetime
print("✅ Libraries imported!")

In [ ]:
import gdown
gdown.download('https://drive.google.com/uc?id=1kd1J5KX5v6FEjr6sahivHFY38BwRJ-r-', 'intents.json', quiet=False)
print("✅ Dataset downloaded!")

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.stem import snowball

snowballStemmer = snowball.SnowballStemmer("english")
print("✅ NLTK ready!")

## Phase 2: Text Preprocessing

In [ ]:
# SOLUTION: Complete text preprocessing function
def text_preprocessing(sentence):
    # Step 1: Tokenize the sentence
    tokens = nltk.word_tokenize(sentence)
    
    # Step 2: Stem each token and filter
    stem_tokens = []
    for token in tokens:
        if token.isalnum():  # Only keep alphanumeric
            stem_tokens.append(snowballStemmer.stem(token.lower()))
    
    # Step 3: Join and return
    return " ".join(stem_tokens)

# Test the function
test_sentence = 'We all agreed, it was a magnificent evening.'
result = text_preprocessing(test_sentence)
print(f"Input: '{test_sentence}'")
print(f"Output: '{result}'")

## Phase 3: Load and Process Dataset

In [ ]:
with open("intents.json") as f:
    data = json.load(f)

intent_list = []
train_data = []
train_label = []
responses = {}

print(f"✅ Loaded {len(data['intents'])} intents")

In [ ]:
# SOLUTION: Process the dataset
for intent in data['intents']:
    intent_list.append(intent['intent'])
    responses[intent['intent']] = intent['responses']
    
    for text in intent['text']:
        preprocessed_text = text_preprocessing(text)
        train_data.append(preprocessed_text)
        train_label.append(intent['intent'])

print(f"Intents: {intent_list}")
print(f"\ntrain_data[2:5]: {train_data[2:5]}")
print(f"train_label[2:5]: {train_label[2:5]}")
print(f"\nTotal training samples: {len(train_data)}")

## Phase 4: Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
print("✅ CountVectorizer ready!")

In [ ]:
# SOLUTION: Create vocabulary and Bag of Words
vectorizer.fit(train_data)

list_of_words = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(list_of_words)}")
print(f"Sample words: {list(list_of_words[:10])}")

train_data_bow = vectorizer.transform(train_data)
print(f"\nBag of Words shape: {train_data_bow.shape}")

print(f"\nOriginal: '{train_data[1]}'")
print(f"Label: {train_label[1]}")

## Phase 5: Train ML Models

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB

# SOLUTION: Train all 3 classifiers
clf_knn = KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(train_data_bow, train_label)
print("✅ KNN trained!")

clf_dt = DecisionTreeClassifier(random_state=33)
clf_dt.fit(train_data_bow, train_label)
print("✅ Decision Tree trained!")

clf_nb = MultinomialNB()
clf_nb.fit(train_data_bow, train_label)
print("✅ Naive Bayes trained!")

## Phase 6: Test Models

In [ ]:
# SOLUTION: Test models
test_sentence = "Hello there"

test_processed = text_preprocessing(test_sentence)
print(f"Preprocessed: '{test_processed}'")

test_bow = vectorizer.transform([test_processed])

print(f"\nKNN predicts: {clf_knn.predict(test_bow)[0]}")
print(f"Decision Tree predicts: {clf_dt.predict(test_bow)[0]}")
print(f"Naive Bayes predicts: {clf_nb.predict(test_bow)[0]}")

In [ ]:
# Test with more sentences
test_sentences = [
    "Thank you so much",
    "What time is it?",
    "Tell me a joke",
    "Goodbye"
]

print("Testing Naive Bayes predictions:")
for sentence in test_sentences:
    processed = text_preprocessing(sentence)
    bow = vectorizer.transform([processed])
    prediction = clf_nb.predict(bow)[0]
    print(f"  '{sentence}' -> {prediction}")

## Phase 7: Chatbot Interface

In [ ]:
# SOLUTION: Complete bot_respond function
def bot_respond(user_query):
    # Step 1: Preprocess user query
    user_query = text_preprocessing(user_query)
    
    # Step 2: Convert to Bag of Words
    user_query_bow = vectorizer.transform([user_query])
    
    # Step 3: Choose classifier (Naive Bayes works best for text)
    clf = clf_nb
    
    # Step 4: Predict intent
    predicted = clf.predict(user_query_bow)
    
    # Handle low confidence predictions
    max_proba = max(clf.predict_proba(user_query_bow)[0])
    if max_proba < 0.08:
        predicted = ['noanswer']
    
    # Step 5: Select random response for predicted intent
    numOfResponses = len(responses[predicted[0]])
    chosenResponse = random.randint(0, numOfResponses-1)
    
    # Handle time query specially (use datetime directly)
    if predicted[0] == "TimeQuery":
        bot_response = f"It's {datetime.now().strftime('%H:%M:%S')}"
    else:
        bot_response = responses[predicted[0]][chosenResponse]
    
    return bot_response

# Test the function
print("Testing bot_respond:")
print(f"  'Hello there' -> {bot_respond('Hello there')}")
print(f"  'What time is it?' -> {bot_respond('What time is it?')}")
print(f"  'Thank you!' -> {bot_respond('Thank you!')}")
print(f"  'Tell me a joke' -> {bot_respond('Tell me a joke')}")

In [ ]:
# Interactive chatbot interface
print("="*50)
print("This is Alex the chatbot. Say something!!")
print("Type 'quit' to exit")
print("="*50)

while True:
    try:
        bot_input = input("You  : ")
        if bot_input.lower() == 'quit':
            print("Alex : Thank you for chatting! See you again soon!!")
            break
        print("Alex :", bot_respond(bot_input))
    except KeyboardInterrupt:
        print("\nAlex : Thank you for choosing us. See you again soon!!")
        break

## 🎉 Project Complete!

Chatbot successfully built with NLP intent classification!

## Extension: Model Accuracy Comparison

In [ ]:
# Compare model accuracy using cross-validation
from sklearn.model_selection import cross_val_score

models = [
    ('KNN', clf_knn),
    ('Decision Tree', clf_dt),
    ('Naive Bayes', clf_nb)
]

print("Model Accuracy Comparison:")
print("-" * 40)
for name, model in models:
    scores = cross_val_score(model, train_data_bow, train_label, cv=3)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")